**Лабораторный практикум по курсу «Распознавание диктора», Университет ИТМО, 2021**		

**Лабораторная работа №2. Обучение детектора речевой активности**

**Цель работы:** изучение процедуры построения энергетического детектора речевой активности на основе модели гауссовых смесей.

**Краткое описание:** в рамках настоящей лабораторной работы предлагается изучить и реализовать энергетический детектор речевой активности на основе модели гауссовых смесей; реализация детектора предполагает субъективную и объективную оценку его качества по целевым метрикам; тестирование детектора выполняется в идеализированных условиях, а также при наличии шумов и помех.

**Данные:** в качестве данных для выполнения лабораторной работы предлагается использовать тестовую часть базы [VoxCeleb1](http://www.robots.ox.ac.uk/~vgg/data/voxceleb/vox1.html).

**Содержание лабораторной работы**

1. Подготовка данных для обучения и тестирования детектора речевой активности.							

2. Изучение файлового формата RTTM.

3. Реализация энергетического детектора речевой активности на основе модели гауссовых смесей.

4. Коррекция разметки детектора речевой активности с использованием методов морфологической обработки.

5. Тестирование реализованного детектора речевой активности на «чистых» данных.

6. Тестирование реализованного детектора речевой активности на искажённых данных.

In [ ]:
# Автоматически перезагружаем изменённые модули перед выполнением кода
%load_ext autoreload
%autoreload 2

# Импортируем необходимые модули
import os
import sys
from math import sqrt
from pathlib import Path

sys.path.append(str(Path('..').resolve()))

import numpy as np
import torchaudio
from matplotlib.pyplot import figure, plot, show, grid, title, xlabel, ylabel, legend, hist
from torchaudio.transforms import Resample

from common import download_dataset, extract_dataset
from constants import (
    AUGMENTATION_AWGN_TEMPLATE,
    AUGMENTATION_CLEAN,
    AUGMENTATION_MODE_AWGN,
    AUGMENTATION_MODE_REVERB,
    AUGMENTATION_REVERB,
    DATA_DIR,
    DATASET_LIST_PATH,
    DEFAULT_RTTM_DIR,
    DEFAULT_VOXCELEB_DIR,
    ERROR_VOXCELEB_CREDENTIALS,
    EXAMPLE_AUDIO_RELATIVE_PATH,
    EXAMPLE_RTTM_FILENAME,
    FRAME_SHIFT,
    FRAME_WINDOW,
    GMM_COMPONENT_LABEL_TEMPLATE,
    GMM_REALIGNMENTS,
    IMPULSE_RESPONSE_PATH,
    LABEL_AMPLITUDE,
    LABEL_DENSITY,
    LABEL_FILTERED_VAD,
    LABEL_GMM_MIXTURE,
    LABEL_IDEAL_VAD,
    LABEL_NORMALIZED_ENERGY,
    LABEL_RAW_VAD,
    LABEL_SAMPLE_NUMBER,
    LABEL_SIGNAL,
    METRICS_MESSAGE,
    MORPHOLOGY_MASK_SIZE,
    NOTEBOOK_AWGN_SIGMA,
    NUMERICAL_EPSILON,
    TEXT_ENCODING,
    TITLE_ENERGY_HISTOGRAM,
    TITLE_GMM_DENSITY,
    TITLE_VAD_MARKUP,
    VAD_THRESHOLD,
    VOXCELEB_ARCHIVE_PATH,
    VOXCELEB_EXTRACT_PATH,
    VOXCELEB_PASSWORD_ENV,
    VOXCELEB_USER_ENV,
)
from exercises_blank import (
    awgn,
    energy_gmm_vad,
    eval_frame_post_prob,
    frame_energy,
    frames_to_samples,
    framing,
    gaussian_pdf,
    gmm_train,
    load_vad_markup,
    morphological_cleanup,
    norm_energy,
    reverb,
)

**1. Подготовка данных для обучения и тестирования детектора речевой активности**

В ходе выполнения лабораторной работы необходимы данные для выполнения процедуры обучения и процедуры тестирования реализуемого детектора речевой активности. Возьмём в качестве этих данных несколько звукозаписей голосов людей мужского и женского пола, сохраненных в формат *wav*, выбранных из корпуса [VoxCeleb1 test set](https://thor.robots.ox.ac.uk/~vgg/data/voxceleb/vox1a/vox1_test_wav.zip). Данный корпус содержит 4,874 звукозаписи (частота дискретизации равна 16кГц) 40 дикторов мужского и женского пола, разговаривающих на английском языке.

В рамках настоящего пункта требуется выполнить загрузку и распаковку звуковых wav-файлов из корпуса VoxCeleb1 test set.

![Рисунок 1](../images/voxceleb_dataset.png "VoxCeleb. Крупномасштабная аудиовизуальная база данных человеческой речи.")

In [ ]:
# Загружаем тестовую часть VoxCeleb1. Данные доступа не храним в ноутбуке.
voxceleb_user = os.environ.get(VOXCELEB_USER_ENV)
voxceleb_password = os.environ.get(VOXCELEB_PASSWORD_ENV)
if not voxceleb_user or not voxceleb_password:
    raise RuntimeError(ERROR_VOXCELEB_CREDENTIALS)

with DATASET_LIST_PATH.open('r', encoding=TEXT_ENCODING) as dataset_list_file:
    dataset_lines = dataset_list_file.readlines()

download_dataset(
    dataset_lines,
    user=voxceleb_user,
    password=voxceleb_password,
    save_path=str(DATA_DIR),
)

In [ ]:
# Распаковываем тестовую часть VoxCeleb1
extract_dataset(
    save_path=str(VOXCELEB_EXTRACT_PATH),
    fname=str(VOXCELEB_ARCHIVE_PATH),
)

**2. Изучение файлового формата RTTM**

*Файлы расширенной транскрипции с метками времени* (RTTM, rich transcription time marked) представляют собой текстовые файлы, в каждой строке которых содержатся разделённые пробелом метаданные, привязанных к звукозаписи объектов. Метаданные аннотируют отдельные элементы звукозаписи. Каждая строка [rttm-файла](./docs/rt09_meeting_eval_plan_v2.pdf) представляет аннотацию одного примера объекта. Типы объектов, привязанных к звукозаписи, варьируются по отношению к решаемой задачи. Так, например, при решении задачи *диаризации* разметка rttm-файла может содержать в каждой строке метаданные, связанные с одним из нескольких дикторов, разговаривающих в звукозаписи. Объектами в этом случае являются дикторы. В случае решения задачи построения детектора речевой активности разметка rttm-файла содержит в каждой строке метаданные, связанные с речевыми сегментами, присутствующими в звукозаписи. Объектом в этом случае является дикторская речь в целом.

Каждая строка rttm-файла содержит 10 полей. Из присутствующих в rttm-файле полей при выполнении настоящей лабораторной работы будут использоваться следующие: *поле 1*, тип объекта (SPEAKER, речь диктора), *поле 2*, имя файла без указания пути до него и расширения, *поле 3*, номер канала, с которым связан речевой сегмент, например, «1» или «2», *поле 4*, время начала речевого сегмента в секундах, *поле 5*, длительность речевого сегмента в секундах. *Поле 6*, *поле 7*, *поле 8*, *поле 9* и *поле 10* будут считаться неопределёнными. Возможный пример части rttm-файла для некоторой звукозаписи представлен ниже.

SPEAKER 00006 1 3.186 1.733 \<NA\> \<NA\> \<NA\> \<NA\> \<NA\>  
SPEAKER 00006 1 5.253 1.043 \<NA\> \<NA\> \<NA\> \<NA\> \<NA\>  
SPEAKER 00006 1 6.376 2.007 \<NA\> \<NA\> \<NA\> \<NA\> \<NA\>  
SPEAKER 00006 1 8.796 0.676 \<NA\> \<NA\> \<NA\> \<NA\> \<NA\>  

В рамках настоящего пункта предлагается проанализировать содержимое директория **./ground_truth/rttm/**, в котором представлены разметки идеального детектора речевой активности в rttm-формате, полученные путём ручной разметки для некоторых звукозаписей из базы данных VoxCeleb1 test set. Настоящие эталонные разметки предполагается использовать для оценки качества реального детектора речевой активности ниже в рамках рассматриваемой лабораторной работы. Отметим, что эталонные разметки звукозаписей на «речь» и «не речь» были получены с использованием [Wave Assistant](https://vk.com/wall-102309764) – программное обеспечение, позволяющее выполнять обработку речевых сигналов. Разметка детектора речевой активности в Wave Assistant может быть сохранена во внутренний seg-формат, из которого, путём несложной конвертации, разметка может быть представлена в rttm-формате.

Идеальные разметки из директория **./ground_truth/rttm/** получены для 20 звукозаписей (10 голосов дикторов женского пола и 10 голос дикторов мужского пола). Имена файлов идеальной разметки имеют следующий формат: **id10271_1gtz-CUIygI_00006.rttm**. В соответстввии со структурой данных базы VoxCeleb1 test set для примера выше это означает, что идеальная разметка, сформирована для звукозаписи **00006.wav**, выбраной из сессии **1gtz-CUIygI** и соответствущей диктору с идентификационным номером **id10271**. Если скачивание и распаковка данных базы VoxCeleb1 test set выполнены корректно в соответствии с пунктом 1 выше, то звукозапись, которой соответствует **./ground_truth/rttm/id10271_1gtz-CUIygI_00006.rttm** может быть найдена по следующему пути: **../data/voxceleb1_test/wav/id10271/1gtz-CUIygI/00006.wav**.

Ниже представлена последовательность действий, которую необходимо выполнить при рассмотрении настоящего пункта:

1. Загрузить и визуализировать осциллограмму любой звукозаписи, идеальная разметка детектора речевой активности которой находится в директории **./ground_truth/rttm/**.

2. Прочитать с использованием функционала базового ядра Python rttm-файл для загруженной звукозаписи.

3. Наложить разметку детектора речевой активности на осциллограмму визуализированной звукозаписи.

In [ ]:
# Указываем пути к аудиозаписи и её эталонной разметке
path_to_wav = DEFAULT_VOXCELEB_DIR / EXAMPLE_AUDIO_RELATIVE_PATH
path_to_rttm = DEFAULT_RTTM_DIR / EXAMPLE_RTTM_FILENAME

# Загружаем сигнал и приводим его к моно
signal, fs = torchaudio.load(str(path_to_wav))
signal = signal.numpy().mean(axis=0)
signal = signal / max(np.abs(signal).max(), NUMERICAL_EPSILON)

# Загружаем эталонную VAD-разметку
vad_markup_ideal = load_vad_markup(path_to_rttm, signal, fs)

# Показываем сигнал и эталонную разметку
figure(figsize=(16, 4))
plot(signal, color='green')
plot(vad_markup_ideal, color='red')
xlabel(LABEL_SAMPLE_NUMBER)
ylabel(LABEL_AMPLITUDE)
title(TITLE_VAD_MARKUP)
grid()
legend([LABEL_SIGNAL, LABEL_IDEAL_VAD])
show()

**3. Реализация энергетического детектора речевой активности на основе модели гауссовых смесей**

*Детектор речевой активности* предназначен для определения присутствия или отсутсвия речи в звукозаписи. Входом детектора является обрабатываемый сигнал, а выходом – разметка на «речь» и «не речь». Представить разметку детектора речевой активности можно различными способами, например, с использованием rttm-файла, путём визуализации в виде последовательности прямоугольных импульсов переменной длительности, наложенных на осциллограмму обрабатываемого сигнала во временной области и т.п. В последнем случае речевые участки обычно маркируется уровнем «1», а неречевые участки – уровнем «0».

На данный момент времени существует множество различных реализаций детекторов речевой активности, в том числе современных, основанных на использовании теории *глубоких нейронных сетей*. Необходимо отметить, что использование детекторов речевой активности является важным при решении задачи *распознавания диктора*, поскольку качество дикторских моделей, формируемых на основе речевых сигналов без длительных пауз, получается значительно лучше. В рамках настоящей лабораторной работы предлагается реализовать *энергетический детектор речевой активности* на основе *модели гауссовых смесей*.

Алгоритм реализации указанного энергетического детектор речевой активности и, фактически, последовательность действий, которую необходимо выполнить в рамках настоящего пункта, представлены ниже:

1. Загрузить речевой сигнал, разметку на «речь» и «не речь» которого необходимо получить.

2. Разбить обрабатываемый речевой сигнал с использованием некоторой оконной функции, например, прямоугольной, на перекрывающиеся сегменты (фреймы или кадры).

3. Вычислить для каждого из сегментов энергию сигнала. Предположив, что длина оконной функции составляет $L$ отсчётов, энергия сигнала в окне может быть вычислена с использованием следующего выражения: 

$$E(i)=\sum_{n = 0}^{L - 1}\left | x(n + i)w(n))) \right |^2,$$

где $x(n)$ – обрабатываемый речевой сигнал, $w(n)$ – оконная функция, а $i$ – шаг окна.

4. Выполнить нормализацию и масштабирование энергий сегментов на среднее значение и среднеквадратическое значение энергии, вычисленные по всем сегментам звукозаписи, получив признаки для обучения модели гауссовых смесей. Вычислить среднее значение и среднеквадратическое значение энергии по всем сегментам обрабатываемого сигнала можно с использованием следующих выражений:

$$m_E = \frac{1}{M}\sum_{i = 0}^{M - 1}E(i), \sigma_E = \sqrt{\frac{1}{M - 1}\sum_{i = 0}^{M - 1}(E(i) - m_E)^{2}},$$

здесь $M$ – количество сегментов, на которые был разбит сигнал. Нормализованные и масштабированные энергии сегментов можно определить следующим образом:

$$E_{norm}(i) = \frac{E(i) - m_E}{\sigma_E}.$$

5. Обучить модель гауссовых смесей для случая трёх одномерных гауссианов с использованием подготовленных выше признаков. В рассматриваемой задаче предполагается, что два из этих гауссианов порождают признаки, соответствующие речевым сегментам, а третий гауссиан порождает признаки, соответствующие неречевым сегментам. Обучение модели гауссовых смесей можно выполнить с использованием [EM-алгоритм](http://www.machinelearning.ru/wiki/index.php?title=EM-%D0%B0%D0%BB%D0%B3%D0%BE%D1%80%D0%B8%D1%82%D0%BC), который, по своей сути, является вероятностным обобщением метода кластеризации на основе [k-средних](https://en.wikipedia.org/wiki/K-means_clustering).

Предположим, что одномерная модель гауссовых смесей из трёх гауссианов на множестве номированных энергий $E_{norm}$ описывается с использованием следующей плотности вероятностей:

$$W_{E_{norm}}(e)=\sum_{j = 0}^{2}w_jW_j(e), \sum_{j = 0}^{2}w_j = 1,$$

где $W_j(e)$ – функция правдоподобия $j$-й компонеты смеси, а $w_j$ – её априорная вероятность. Каждая компонента смеси может быть описана гауссовой плотностью вероятностей:

$$W_j(e) = N(e; m_j, \sigma_j) =\frac{1}{\sigma_j\sqrt{2\pi}}e^{\frac{-(e - m_j)^2}{2\sigma_j^2}},$$

где $m_j$ и $\sigma_j$ – это математическое ожидание и среднеквадратическое отклонение соответствующих компонент гауссовой смеси, а $j = 0, 1, 2$.

*E-шаг* алгоритма обучения модели гауссовых смесей можно описать так:

$$g_{ij} = \frac{w_jN(E_{norm}(i); m_j, \sigma_j)}{\sum_{s = 0}^{2}w_sN(E_{norm}(i); m_s, \sigma_s)}.$$

*M-шаг* алгоритма обучения модели гауссовых смесей можно описать с использованием следующих выражений:

$$w_j = \frac{1}{M}\sum_{i = 0}^{M - 1}g_{ij}, m_j = \frac{1}{Mw_j}\sum_{i = 0}^{M - 1}g_{ij}E_{norm}(i), \sigma_j = \sqrt{\frac{1}{Mw_j}\sum_{i = 0}^{M - 1}g_{ij}(E_{norm}(i) - m_j)^2},$$

здесь $M$ – количество сегментов, на которые был разбит сигнал.

6. С использованием обученной модели оценить принадлежность сегмента к «не речи». В предположении, что значения энергий неречевых сегментов порождаются гауссианом с индексом «0» оценим апостериорные вероятности того, что объект $E_{norm}(i)$ получен из 0-й компоненты смеси:

$$\hat{g}_{i0} = \frac{\hat{w}_0N(E_{norm}(i); \hat{m}_0, \hat{\sigma}_0)}{\sum_{s = 0}^{2}\hat{w}_sN(E_{norm}(i); \hat{m}_s, \hat{\sigma}_s)},$$

здесь $\hat{w}_j$, $\hat{m}_j$ и $\hat{\sigma}_j$ являются обученными параметрами модели гауссовых смесей. Чем выше $\hat{g}_{i0}$ для выбранного значения $E_{norm}(i)$, тем наиболее вероятно, что соответствующий фрейм относится к неречевому.

7. Подобрать порог принятия решения для детектора речевой активности.

8. Сформировать пофреймовую разметку детектора речевой активности на «речь» и «не речь», маркируя речевые фреймы символом «1»,а не речевые – символом «0». Выполнить данную процедуру можно путём сравнения величины $\hat{g}_{i0}$ для выбранного значения $E_{norm}(i)$ с заданным порогом детектора. Если величина $\hat{g}_{i0}$ находится ниже порога, то сегмент предполагается речевым, в противном случае – неречевым. В случае необходимости пофреймовая разметка детектора речевой активности может быть преобразована в разметку по отсчётам сигнала. Последнее позволит, например, визуально сравнить реальную и эталонную разметки детектора путём их наложения на осциллограмму обрабатываемой звукозаписи.

In [ ]:
# Возводим отсчёты сигнала в квадрат: так получаем мгновенную энергию
squared_signal = signal ** 2

In [ ]:
# Разбиваем сигнал на перекрывающиеся фреймы
window = FRAME_WINDOW
shift = FRAME_SHIFT
frames = framing(squared_signal, window=window, shift=shift)

In [ ]:
# Суммируем квадраты отсчётов внутри каждого фрейма
energy = frame_energy(frames)

In [ ]:
# Нормализуем энергию, чтобы среднее стало близким к 0, а отклонение — к 1
normalized_energy = norm_energy(energy)

# Строим гистограмму энергий фреймов
hist(
    normalized_energy,
    int(sqrt(len(normalized_energy))),
    histtype='step',
    color='green',
)
xlabel(LABEL_NORMALIZED_ENERGY)
ylabel(LABEL_DENSITY)
title(TITLE_ENERGY_HISTOGRAM)
grid()
show()

In [ ]:
# Используем готовую функцию плотности нормального распределения
gauss_pdf = gaussian_pdf

In [ ]:
# Обучаем параметры смеси гауссовых распределений EM-алгоритмом
weights, means, sigmas = gmm_train(
    normalized_energy,
    gauss_pdf,
    n_realignment=GMM_REALIGNMENTS,
)

sorted_energy = np.sort(normalized_energy)
mixture_density = np.zeros(len(normalized_energy))
for component_index in range(len(means)):
    component_density = (
        weights[component_index]
        * gauss_pdf(sorted_energy, means[component_index], sigmas[component_index])
    )
    mixture_density += component_density
    plot(
        sorted_energy,
        component_density,
        label=GMM_COMPONENT_LABEL_TEMPLATE.format(index=component_index + 1),
    )

# Показываем компоненты GMM и их сумму
plot(
    sorted_energy,
    mixture_density,
    color='black',
    linewidth=2,
    label=LABEL_GMM_MIXTURE,
)
xlabel(LABEL_NORMALIZED_ENERGY)
ylabel(LABEL_DENSITY)
title(TITLE_GMM_DENSITY)
grid()
legend()
show()

In [ ]:
# Оцениваем вероятность того, что каждый фрейм относится к паузе
silence_probability = eval_frame_post_prob(
    normalized_energy,
    gauss_pdf,
    weights,
    means,
    sigmas,
)

In [ ]:
# Если вероятность паузы меньше порога, считаем фрейм речевым
vad_frame_markup_real = (silence_probability < VAD_THRESHOLD).astype(np.float32)

# Переводим пофреймовую разметку в посэмпловую
vad_markup_real = frames_to_samples(
    vad_frame_markup_real,
    len(signal),
    FRAME_SHIFT,
)

# Показываем исходный результат детектора до морфологической обработки
figure(figsize=(16, 4))
plot(signal, color='green')
plot(vad_markup_real, color='blue')
xlabel(LABEL_SAMPLE_NUMBER)
ylabel(LABEL_AMPLITUDE)
title(TITLE_VAD_MARKUP)
grid()
legend([LABEL_SIGNAL, LABEL_RAW_VAD])
show()

**4. Коррекция разметки детектора речевой активности с использованием методов морфологической обработки**

Анализ разметки, полученной с использованием детектора речевой активности выше показывает, что она сильно отличается от эталонной разметки высокой частотой нарезки звукозаписи на речевые сегменты. Попытаться сделать разметку детектора менее частой, в случае необходимости, можно с использованием методов [морфологической фильтрации](http://www.theobjects.com/dragonfly/dfhelp/4-0/Content/05_Image%20Processing/Morphology%20Filters.htm), применяемых в ранних системах обработки цифровых изображений, в первую очередь *бинарных*, состоящих только из пикселей двух уровней яркости. Фактически, разметку детектора речевой активности можно рассматривать в качестве одной строки или столбца бинарного цифрового изображения. Поэтому с использованием некоторой одномерной бинарной маски *морфологического фильтра*, называемой *структурирующим элементом*, разметка детектора речевой активности может быть отфильтрована.

Ключевыми процедурами *морфологической обработки* являются следующие:

1. *Наращивание (dilate)* – процедура морфологичекой фильтрации, позволяющая увеличить область цифрового изображения.
2. *Эрозия (erode)* – процедура морфологичекой фильтрации, позволяющая уменьшить область цифрового изображения.
3. *Замыкание (close)* – процедура морфологичекой фильтрации, представляющая последовательное использование наращивания и эрозии.
4. *Размыкание (open)* – процедура морфологичекой фильтрации, представляющая последовательное использование эрозии и наращивания.

В рамках настоящего пункта предлагается провести эксперименты с процедурами замыкания и размыкания применительно к полученной выше разметке детектора речевой активности.

In [ ]:
# Закрытие заполняет короткие разрывы, а открытие удаляет короткие выбросы
vad_markup_real_filtered = morphological_cleanup(
    vad_markup_real,
    MORPHOLOGY_MASK_SIZE,
)

# Сравниваем эталонную, исходную и обработанную разметки
figure(figsize=(16, 4))
plot(signal, color='green')
plot(vad_markup_ideal, color='red')
plot(vad_markup_real_filtered, color='blue')
xlabel(LABEL_SAMPLE_NUMBER)
ylabel(LABEL_AMPLITUDE)
title(TITLE_VAD_MARKUP)
grid()
legend([LABEL_SIGNAL, LABEL_IDEAL_VAD, LABEL_FILTERED_VAD])
show()

**5. Тестирование реализованного детектора речевой активности на «чистых» данных**

Визульная оценка качества работы детектора речевой активности является нагдядной, но не даёт глобального понимания того, насколько хорошо работает детектор речевой активности «глобально» в рамках некоторого кейса и не позволяет эффективно сравнить работу одного детектора с другим. Для решения указанной проблемы требуется использовать объективные метрики оценки качества. Поскольку результатом детектора речевой активности является разделение сигнала на участки двух типов, «речь» и «не речь», то для численной оченки качества в данном случае подойдут метрики, используемые в бинарной теории принятия решений. К ним можно отнести *вероятности ошибок первого (FRR, false rejection rate, или FNR, false negative rate)* и *второго рода (FAR, false acceptance rate, или FPR, false positive rate)*, *точность (P, precision)* и *полнота выборки (R, recall)*, *равный уровень ошибок (EER, equal error rate)* и т.п. Идеи реализаций некоторых из указанных метрик, применительно к оценке качества результатов бинарной сегментации цифровых изображений, могут быть найдены по следующей [ссылке](https://github.com/DeepTrial/AutoMetric/blob/master/autoMetric.py).

В рамках настоящего пункта требуется вычислить вероятности ошибок первого и второго рода, а также точность и полноту выборки для фиксированного значения порога детектора речевой активности. При вычислении метрик предположим, что *нулевой гипотезой* является наличие речи, а альтернативной – отсутствие речи в некотором сегменте обрабатываемого сигнала. В качестве данных для оенки качества детектора речевой активности воспользуемся 20 звукозаписями, разметка идеального детектора которых представлена в директории **./ground_truth/rttm/**. 

In [ ]:
def evaluate_voxceleb(augmentation_mode=None):
    """Оценить VAD на всех имеющихся размеченных записях VoxCeleb."""
    true_positive = false_positive = false_negative = true_negative = 0.0

    for rttm_file in sorted(DEFAULT_RTTM_DIR.glob('*.rttm')):
        speaker_id, remainder = rttm_file.stem.split('_', 1)
        video_id, file_id = remainder.rsplit('_', 1)
        audio_path = DEFAULT_VOXCELEB_DIR / speaker_id / video_id / f'{file_id}.wav'

        current_signal, current_fs = torchaudio.load(str(audio_path))
        current_signal = current_signal.numpy().mean(axis=0)
        current_signal /= max(np.abs(current_signal).max(), NUMERICAL_EPSILON)

        if augmentation_mode == AUGMENTATION_MODE_REVERB:
            impulse_response, impulse_fs = torchaudio.load(str(IMPULSE_RESPONSE_PATH))
            resample = Resample(orig_freq=impulse_fs, new_freq=current_fs)
            impulse_response = resample(impulse_response).numpy().mean(axis=0)
            impulse_response /= max(
                np.abs(impulse_response).max(),
                NUMERICAL_EPSILON,
            )
            current_signal = reverb(current_signal, impulse_response)
        elif augmentation_mode == AUGMENTATION_MODE_AWGN:
            current_signal = awgn(current_signal, NOTEBOOK_AWGN_SIGMA)

        ideal_markup = load_vad_markup(
            rttm_file,
            current_signal,
            current_fs,
        )
        real_markup = energy_gmm_vad(
            current_signal,
            FRAME_WINDOW,
            FRAME_SHIFT,
            gaussian_pdf,
            GMM_REALIGNMENTS,
            VAD_THRESHOLD,
            MORPHOLOGY_MASK_SIZE,
        )

        true_positive += np.sum(ideal_markup * real_markup)
        false_positive += np.sum((1 - ideal_markup) * real_markup)
        false_negative += np.sum(ideal_markup * (1 - real_markup))
        true_negative += np.sum((1 - ideal_markup) * (1 - real_markup))

    def safe_divide(numerator, denominator):
        return numerator / denominator if denominator else 0.0

    fnr = safe_divide(false_negative, false_negative + true_positive)
    fpr = safe_divide(false_positive, false_positive + true_negative)
    precision = safe_divide(true_positive, true_positive + false_positive)
    recall = safe_divide(true_positive, true_positive + false_negative)
    return fnr, fpr, precision, recall


fnr, fpr, precision, recall = evaluate_voxceleb()
print(
    METRICS_MESSAGE.format(
        name=AUGMENTATION_CLEAN,
        fnr=fnr,
        fpr=fpr,
        precision=precision,
        recall=recall,
    )
)

**6. Тестирование реализованного детектора речевой активности на искажённых данных**

Процедуры формирования и передачи речевого сигнала могут сопровождаться воздействием шумов и помех, приводящих к искажению сигнала. В качестве примеров искажающих факторов, влияющих на ухудшение качестве речевого сигнала можно привести: импульсный отклик помещения (реверберация), фоновый шум голосов группы нецелевых дикторов, звук телевизора или радиоприёмника и т.п. Разработка конвейера системы голосовой биометрии требует учёта воздействия искажающих факторов на качество работы отдельных блоков и всей системы в целом. Поскольку отдельные этапы современных систем голосовой биометрии реализуются с использованием глубоких нейронных сетей, требующих большие объёмы данных для обучения, то возможным вариантом учёта искажений, воздействующих на систему, может являться расширение тренировочной выборки для обучения моделей за счёт методов аугментации. *Аугментация* – методика создания дополнительных обучающих примеров из имеющихся данных. Как правило, при решении задачи аугментации данных в речевой обработке используются дополнительные базы шумов и помех. В качестве примеров можно привести базы [SLR17](https://openslr.org/17/) (корпус музыкальных, речевых и шумовых звукозаписей) и [SLR28](https://openslr.org/28/) (база данных реальных и симулированных импульсных откликов комнат, а также изотропных и точечных шумов).

Рассмотрим в рамках настоящего пункта техники аугментации, моделирующие воздествие *реверберации* и *аддитивного белого гауссовского шума (АБГШ)* на речевой сигнал и проанализируем их влияние на качество работы детектора речевой активности. В рамках настоящего пункта требуется:

1. Промоделировать эффект реверберации для некоторого записанного импульсного отклика помещения с использованием *линейного инвариантного к сдвигу фильтра*. При моделировании эффекта реверберации предлагается воспользоваться импульсным откликом [Лозаннского собора](https://en.wikipedia.org/wiki/Lausanne_Cathedral), который измерили Докманич И. и его коллеги, записавшие звук лопнувшего в соборе воздушного шарика. 

2. Промоделировать воздействие аддитивного шума с использованием модели АБГШ для различных уровней мощности шума.

3. Выполнить тестирование детектора речевой активности при наличии реверберации или АБГШ с использованием описанных выше численных метрик оценки качества.

In [ ]:
# Выбираем один из режимов: реверберация или аддитивный белый гауссов шум
augmentation_mode = AUGMENTATION_MODE_AWGN

fnr, fpr, precision, recall = evaluate_voxceleb(augmentation_mode)
if augmentation_mode == AUGMENTATION_MODE_REVERB:
    augmentation_name = AUGMENTATION_REVERB
else:
    augmentation_name = AUGMENTATION_AWGN_TEMPLATE.format(
        sigma=NOTEBOOK_AWGN_SIGMA,
    )

print(
    METRICS_MESSAGE.format(
        name=augmentation_name,
        fnr=fnr,
        fpr=fpr,
        precision=precision,
        recall=recall,
    )
)

**7. Контрольные вопросы**

1. Как устроен rttm-файл и для каких целей он используется?
2. Как вычислить энергию сигнала?
3. Что такое модель гауссовых смесей?
4. Что такое детектор речевой активности?
5. Как устроен энергетический детектор речевой активности на основе модели гауссовых смесей?
6. Основные достоинства и недостатки энергетического детектора речевой активности.
7. В чём идея использования методов морфологической обработки в коррекции разметки детектора речевой активности?
8. Какие численные метрики оценки качества можно использовать для описания результата работы детектора речевой активности?
9. Что такое аугментация данных и для каких целей она используется?
10. Как можно построить более совершенный, чем энергетический, детектор речевой активности?

**8. Список литературы**

1. Bishop C.M. Pattern recognition and machine learning. Springer, 2006 ([ссылка](http://users.isr.ist.utl.pt/~wurmd/Livros/school/Bishop%20-%20Pattern%20Recognition%20And%20Machine%20Learning%20-%20Springer%20%202006.pdf)).
2. Beigi H. Fundamentals of speaker recogniton. Springer, 2011 ([ссылка](https://www.springer.com/gp/book/9780387775913)).
3. Černocký H. Voice biometry standard proposal // Interspeech 2015, Dresden, Germany, 2015 ([ссылка](https://voicebiometry.org/)).